In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
df = pd.read_csv(r"C:\Users\Ahmed\Downloads\supermarket_sales -.csv")
df.head()

In [ ]:
print("Missing values:\n", df.isnull().sum())

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
X = df.drop('Customer type', axis=1)  # x= ?? ???????? ????? ?????? ???? ????? ????? ????
y = df['Customer type']
y

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()

X_encoded = X_encoded.drop('Invoice ID', axis=1)

le = LabelEncoder()
for col in X_encoded.columns:
    if X_encoded[col].dtype == 'object':
        X_encoded[col] = le.fit_transform(X_encoded[col])


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training Is : {len(X_train)}")
print(f"Validation Is : {len(X_val)}")
print(f"Test Is : {len(X_test)}")

In [ ]:
#KNN
k_values = range(1, 21)
val_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    val_score = knn.score(X_val, y_val)
    val_scores.append(val_score)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(k_values, val_scores, marker='x')
plt.xlabel('K value')
plt.ylabel('Validation Accuracy')
plt.title('Validation Accuracy for different K')
plt.show()

In [ ]:
best_k = k_values[np.argmax(val_scores)]
print(f"Best K: {best_k}")

In [ ]:
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

In [ ]:
# Accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {test_accuracy}")

In [ ]:
# 6. Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(KNeighborsClassifier(n_neighbors=best_k), X_train, y_train, cv=kf)

print(f"Cross-validation mean accuracy: {cv_scores}")
print(f"Cross-validation mean accuracy: {cv_scores.mean()}")

In [ ]:
# 7. Confusion Matrix 
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
report = classification_report(y_test, y_pred)
print(report)